<a href="https://colab.research.google.com/github/drperezt/GCP_devs/blob/main/Pipeline_ETL_BANXICO_EEEESP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalar librerías para manejo de PDFs, solicitudes HTTP y cliente de GCP
!pip install pdfplumber requests google-cloud-bigquery google-cloud-storage pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.0 MB/s eta 0:00:00


In [11]:
import os
import requests
from bs4 import BeautifulSoup

# URL de la sección de encuestas de Banxico
URL_BANXICO = "https://www.banxico.org.mx/publicaciones-y-prensa/encuestas-sobre-las-expectativas-de-los-especialis/encuestas-expectativas-del-se.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

def descargar_ultimo_pdf_texto_completo():
    print("Accediendo al portal de publicaciones de Banxico...")
    response = requests.get(URL_BANXICO, headers=headers)

    if response.status_code != 200:
        print(f"Error al acceder a la página ({response.status_code})")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    link_pdf = None

    # Buscar el primer enlace ('a') cuyo texto contenga "Texto completo"
    for a in soup.find_all('a', href=True):
        texto_enlace = a.get_text(strip=True).lower()
        if "texto completo" in texto_enlace and '.pdf' in a['href'].lower():
            link_pdf = a['href']
            print(f"✅ Enlace 'Texto completo' encontrado: {link_pdf}")
            break

    if not link_pdf:
        print("❌ No se encontró ningún enlace con el texto 'Texto completo'.")
        return None

    # Completar la URL si es relativa
    if not link_pdf.startswith('http'):
        link_pdf = "https://www.banxico.org.mx" + link_pdf

    print(f"Descargando PDF desde: {link_pdf}")
    pdf_res = requests.get(link_pdf, headers=headers)

    if pdf_res.status_code == 200:
        pdf_filename = "encuesta_banxico.pdf"
        with open(pdf_filename, "wb") as f:
            f.write(pdf_res.content)
        print("🚀 PDF guardado exitosamente como:", pdf_filename)
        return pdf_filename
    else:
        print(f"❌ Error al descargar el PDF. Código HTTP: {pdf_res.status_code}")
        return None

pdf_path = descargar_ultimo_pdf_texto_completo()

Accediendo al portal de publicaciones de Banxico...
✅ Enlace 'Texto completo' encontrado: /publicaciones-y-prensa/encuestas-sobre-las-expectativas-de-los-especialis/{9A769BA5-F259-4032-8399-BAE68C36ABFA}.pdf
Descargando PDF desde: https://www.banxico.org.mx/publicaciones-y-prensa/encuestas-sobre-las-expectativas-de-los-especialis/{9A769BA5-F259-4032-8399-BAE68C36ABFA}.pdf
🚀 PDF guardado exitosamente como: encuesta_banxico.pdf


In [18]:
import pdfplumber
import pandas as pd
import re

def extraer_expectativas_completas(pdf_file):
    registros_cuadro1 = []
    registros_cuadro2 = []

    with pdfplumber.open(pdf_file) as pdf:
        for page_num, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text:
                continue

            lines = text.split("\n")

            # --- PARSEO CUADRO 1 ---
            if "Cuadro 1" in text:
                indicador_actual = None
                for line in lines:
                    line_clean = line.strip()
                    # Detectar encabezado de indicador
                    if any(kw in line_clean.lower() for kw in [
                        "tipo de cambio", "tasa de fondeo", "inflación general",
                        "inflación subyacente", "crecimiento del pib"
                    ]):
                        indicador_actual = line_clean
                        continue

                    if indicador_actual and "expectativa para" in line_clean.lower():
                        match_anio = re.search(r'expectativa para (\d{4})', line_clean, re.IGNORECASE)
                        numeros = re.findall(r'\b\d+\.\d+\b', line_clean)

                        if match_anio and len(numeros) >= 4:
                            # Guardamos en formato estructurado
                            registros_cuadro1.append({
                                'Cuadro': 'Cuadro 1. Principales Indicadores',
                                'Indicador': indicador_actual,
                                'Horizonte': match_anio.group(1),
                                'Media_Junio': float(numeros[0]),
                                'Media_Julio': float(numeros[1]),
                                'Mediana_Junio': float(numeros[2]),
                                'Mediana_Julio': float(numeros[3])
                            })

            # --- PARSEO CUADRO 2 ---
            if "Cuadro 2" in text:
                periodo_actual = None
                for line in lines:
                    line_clean = line.strip()
                    if line_clean.lower().startswith("para"):
                        periodo_actual = re.sub(r'\s*\d+/', '', line_clean)
                        continue

                    if periodo_actual and re.search(r'^(media|mediana)', line_clean, re.IGNORECASE):
                        match_tipo = re.search(r'^(media|mediana)', line_clean, re.IGNORECASE)
                        tipo_est = match_tipo.group(1).capitalize()
                        numeros = re.findall(r'\b\d+\.\d+\b', line_clean)

                        if len(numeros) >= 4:
                            registros_cuadro2.append({
                                'Cuadro': 'Cuadro 2. Expectativas Inflación',
                                'Indicador': f"Inflación ({periodo_actual})",
                                'Estadistico': tipo_est,
                                'Inf_General_Junio': float(numeros[0]),
                                'Inf_General_Julio': float(numeros[1]),
                                'Inf_Subyacente_Junio': float(numeros[2]),
                                'Inf_Subyacente_Julio': float(numeros[3])
                            })

    df1 = pd.DataFrame(registros_cuadro1)
    df2 = pd.DataFrame(registros_cuadro2)

    return df1, df2

# Ejecutar sobre el PDF descargado
df_c1, df_c2 = extraer_expectativas_completas(pdf_path)

print("=== CUADRO 1: PRINCIPALES INDICADORES ===")
display(df_c1.head(10))

print("\n=== CUADRO 2: EXPECTATIVAS DE INFLACIÓN ANUAL ===")
display(df_c2.head(10))

=== CUADRO 1: PRINCIPALES INDICADORES ===


,Cuadro,Indicador,Horizonte,Media_Junio,Media_Julio,Mediana_Junio,Mediana_Julio
0,Cuadro 1. Principales Indicadores,Expectativa para 2026 4.21 3.92 4.20 4.00  La...,2027,3.86,3.85,3.84,3.84
1,Cuadro 1. Principales Indicadores,Inflación Subyacente (dic.-dic.) de 2026 y 202...,2026,4.17,4.05,4.18,4.00
2,Cuadro 1. Principales Indicadores,Inflación Subyacente (dic.-dic.) de 2026 y 202...,2027,3.80,3.84,3.80,3.86
3,Cuadro 1. Principales Indicadores,Crecimiento del PIB (Δ% anual),2026,1.07,1.11,1.10,1.20
4,Cuadro 1. Principales Indicadores,Crecimiento del PIB (Δ% anual),2027,1.76,1.79,1.80,1.80
5,Cuadro 1. Principales Indicadores,Tipo de Cambio Pesos/Dólar (cierre de año),2026,17.83,17.82,17.88,17.88
6,Cuadro 1. Principales Indicadores,Tipo de Cambio Pesos/Dólar (cierre de año),2027,18.40,18.38,18.50,18.45
7,Cuadro 1. Principales Indicadores,Tasa de fondeo interbancario (cierre de año),2026,6.50,6.49,6.50,6.50
8,Cuadro 1. Principales Indicadores,Tasa de fondeo interbancario (cierre de año),2027,6.47,6.49,6.50,6.50



=== CUADRO 2: EXPECTATIVAS DE INFLACIÓN ANUAL ===


,Cuadro,Indicador,Estadistico,Inf_General_Junio,Inf_General_Julio,Inf_Subyacente_Junio,Inf_Subyacente_Julio
0,Cuadro 2. Expectativas Inflación,Inflación (Para 2026 (dic.-dic.)),Media,4.21,3.92,4.17,4.05
1,Cuadro 2. Expectativas Inflación,Inflación (Para 2026 (dic.-dic.)),Media,4.20,4.00,4.18,4.00
2,Cuadro 2. Expectativas Inflación,Inflación (Para los próximos 12 meses probabil...,Media,3.93,4.15,3.85,3.89
3,Cuadro 2. Expectativas Inflación,Inflación (Para los próximos 12 meses probabil...,Media,3.95,4.18,3.89,3.87
4,Cuadro 2. Expectativas Inflación,Inflación (Para 2027 (dic.-dic.)),Media,3.86,3.85,3.80,3.84
5,Cuadro 2. Expectativas Inflación,Inflación (Para 2027 (dic.-dic.)),Media,3.84,3.84,3.80,3.86
6,Cuadro 2. Expectativas Inflación,Inflación (Para 2028 (dic.-dic.) general corre...,Media,3.78,3.76,3.73,3.75
7,Cuadro 2. Expectativas Inflación,Inflación (Para 2028 (dic.-dic.) general corre...,Media,3.78,3.75,3.72,3.75


In [21]:
import os
from google.colab import auth
from google.cloud import bigquery
from google.oauth2 import service_account
import pandas as pd
from datetime import datetime

# --- CONFIGURACIÓN DE CREDENCIALES Y GCP ---
# 1. Ruta de tu archivo de credenciales Service Account JSON
#JSON_KEY_PATH = "mi-clave-gcp.json"  # Cambia por el nombre exacto de tu archivo JSON

# 2. Parámetros de tu entorno GCP
PROJECT_ID = "project-4e4ef09d-ad1f-4ad6-963"     # Reemplaza con tu Project ID de GCP
DATASET_ID = "data_warehouse_finance"           # Nombre de tu Dataset en BigQuery
TABLE_ID = "expectativas_macro"       # Nombre de la Tabla

# Cargar las credenciales
#credentials = service_account.Credentials.from_service_account_file(JSON_KEY_PATH)
#client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

# --- PREPARACIÓN Y NORMALIZACIÓN DE DATOS ---
# Añadir metadatos de auditoría (Fecha de ejecución)
fecha_ejecucion = datetime.now().strftime("%Y-%m-%d")

# Transformar Cuadro 1 a formato común
df_c1_bq = df_c1.rename(columns={
    'Cuadro': 'cuadro_origen',
    'Indicador': 'indicador',
    'Horizonte': 'periodo_objetivo',
    'Media_Junio': 'media_mes_anterior',
    'Media_Julio': 'media_mes_actual',
    'Mediana_Junio': 'mediana_mes_anterior',
    'Mediana_Julio': 'mediana_mes_actual'
})
df_c1_bq['estadistico'] = 'Ambos (Media/Mediana)'

# Transformar Cuadro 2 a formato común
df_c2_bq = df_c2.rename(columns={
    'Cuadro': 'cuadro_origen',
    'Indicador': 'indicador',
    'Estadistico': 'estadistico',
    'Inf_General_Junio': 'media_mes_anterior',  # O columna según corresponda
    'Inf_General_Julio': 'media_mes_actual',
    'Inf_Subyacente_Junio': 'mediana_mes_anterior',
    'Inf_Subyacente_Julio': 'mediana_mes_actual'
})
df_c2_bq['periodo_objetivo'] = 'Anual'

# Concatenar ambos DataFrames en una sola tabla unificada
df_final = pd.concat([df_c1_bq, df_c2_bq], ignore_index=True)
df_final['fecha_proceso'] = fecha_ejecucion

# --- CARGA A BIGQUERY ---
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

job_config = bigquery.LoadJobConfig(
    # WRITE_APPEND agrega nuevas filas; WRITE_TRUNCATE sobrescribe la tabla
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    # Autodetección de esquema basada en los tipos de datos de pandas
    autodetect=True
)

print(f"Cargando {len(df_final)} registros a {table_ref}...")

job = client.load_table_from_dataframe(
    df_final, table_ref, job_config=job_config
)

# Esperar a que el Job de carga termine
job.result()

print(f"🚀 ¡Carga exitosa! La tabla {TABLE_ID} se ha actualizado correctamente en BigQuery.")

Cargando 17 registros a project-4e4ef09d-ad1f-4ad6-963.data_warehouse_finance.expectativas_macro...
🚀 ¡Carga exitosa! La tabla expectativas_macro se ha actualizado correctamente en BigQuery.
